In [3]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
import time

c:\My_Space\Repos\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [4]:
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    step3: str

In [5]:
def step1(state: CrashState) -> CrashState:
    print("Step 1 Executed..!")
    return {"step1":"done", "input": state.get("input")}

def step2(state: CrashState) -> CrashState:
    print("Step 2 hanging.. Now manually interrupt from the notebook toolbar (STOP Button.)")
    time.sleep(30)
    return {"step2":"done"}

def step3(state: CrashState) -> CrashState:
    print("Step 3 Executed..!")
    return {"step3":"done"}

In [6]:
graph = StateGraph(CrashState)

graph.add_node("step1",step1)
graph.add_node("step2",step2)
graph.add_node("step3",step3)

graph.add_edge(START,"step1")
graph.add_edge("step1","step2")
graph.add_edge("step2","step3")
graph.add_edge("step3",END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [7]:
try:
    print(f"Running graph: Please manullay interrupt during step2..!")
    config = {"configurable":{"thread_id":"1"}}
    initial_state = {"input": "start"}
    final_state = workflow.invoke(initial_state,config=config)
    print(final_state)
except KeyboardInterrupt:
    print("Kernel manually interrupted. (crash simulated)")

Running graph: Please manullay interrupt during step2..!
Step 1 Executed..!
Step 2 hanging.. Now manually interrupt from the notebook toolbar (STOP Button.)
Kernel manually interrupted. (crash simulated)


In [9]:
workflow.get_state(config=config)

StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128544-e06a-672b-8001-aa975858fe22'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-03-25T14:09:57.661871+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128544-e068-641e-8000-9627c637889f'}}, tasks=(PregelTask(id='d04072ed-79cb-5edd-ef57-0d5b8d616739', name='step2', path=('__pregel_pull', 'step2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [11]:
list(workflow.get_state_history(config=config))

[StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128544-e06a-672b-8001-aa975858fe22'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-03-25T14:09:57.661871+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128544-e068-641e-8000-9627c637889f'}}, tasks=(PregelTask(id='d04072ed-79cb-5edd-ef57-0d5b8d616739', name='step2', path=('__pregel_pull', 'step2'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'start'}, next=('step1',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128544-e068-641e-8000-9627c637889f'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-03-25T14:09:57.660973+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128544-e065-6d3

In [12]:
try:
    print(f"Running graph: Please manullay interrupt during step2..!")
    config = {"configurable":{"thread_id":"1"}}
    final_state = workflow.invoke(None,config=config)
    print(final_state)
except KeyboardInterrupt:
    print("Kernel manually interrupted. (crash simulated)")

Running graph: Please manullay interrupt during step2..!
Step 2 hanging.. Now manually interrupt from the notebook toolbar (STOP Button.)
Step 3 Executed..!
{'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}


In [16]:
workflow.get_state(config=config)
list(workflow.get_state_history(config=config))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12854c-2cbe-6122-8003-bbe6fb01d917'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-03-25T14:13:13.570122+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12854c-2cbc-6cf8-8002-383bd0e39158'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step3',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12854c-2cbc-6cf8-8002-383bd0e39158'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-25T14:13:13.569598+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128544-e06a-672b-8001-aa975858fe22'}}, tasks=(PregelTask(id='e92810b0-0045-9511-d5fe-fe520deb3345', name='st